# EDA — Telco Customer Churn Dataset

**Dataset:** 7,043 khách hàng, 21 cột (sau khi loại customerID: 20 cột). Target nhị phân Churn (Yes/No).

- `SeniorCitizen` (0/1) phải được xử lý là **categorical**, không phải continuous — nếu không model sinh sẽ ra giá trị 0.3 vô nghĩa
- `TotalCharges` có 11 dòng trống (tenure=0, khách mới chưa phát sinh cước) — cần convert string → numeric
- `TotalCharges` ≈ `tenure` × `MonthlyCharges` — **functional dependency** cực kỳ quan trọng cho Fidelity

In [1]:
import os
import sys

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.insert(0, '.')

import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings('ignore')

from eda_framework.core.statistics import (
    cramers_v, correlation_ratio, theils_u, shannon_entropy,
    mutual_information, partial_correlation,
    describe_continuous, describe_categorical,
    normality_test, multimodality_kde_peaks,
    detect_outliers_iqr, detect_outliers_zscore, detect_outliers_mad,
    missing_analysis, duplicate_analysis, cardinality_analysis,
    vif_analysis, class_balance, covariate_shift_detection,
)

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.figsize':(10,6),'figure.dpi':100})

In [2]:
df = pd.read_csv('data/Telco-Customer-Churn.csv')
# TotalCharges: string -> numeric, coerce NaN cho 11 dòng trống
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

c_cont = ['tenure','MonthlyCharges','TotalCharges']
c_cat  = ['gender','SeniorCitizen','Partner','Dependents','PhoneService',
          'MultipleLines','InternetService','OnlineSecurity','OnlineBackup',
          'DeviceProtection','TechSupport','StreamingTV','StreamingMovies',
          'Contract','PaperlessBilling','PaymentMethod']
target = 'Churn'
print("Shape:", df.shape)
print("Continuous:", c_cont)
print("Categorical:", len(c_cat), "cột")
print("Target:", df['Churn'].value_counts().to_dict())
print("Tỷ lệ Churn=Yes:", f"{df['Churn'].value_counts(normalize=True)['Yes']*100:.1f}%")

Shape: (7043, 21)
Continuous: ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical: 16 cột
Target: {'No': 5174, 'Yes': 1869}
Tỷ lệ Churn=Yes: 26.5%


---
## Bước 1: Cấu trúc, Chất lượng & Phân phối từng biến

**Mục đích:** Xác định dung lượng dữ liệu, missing, duplicate, và các đặc điểm phân phối giúp dự đoán mức độ khó cho mô hình sinh.

**Góc nhìn khoa học dữ liệu:**
- `TotalCharges` bị missing 11 dòng (0.16%) — rất ít, có thể median imputation
- `SeniorCitizen` là binary (0/1) với 83.8% là 0 — **mất cân bằng nghiêm trọng**, model sinh dễ bỏ qua lớp thiểu số
- `tenure` có phân phối gần uniform (khách mới và khách lâu năm đều có) — dễ tái tạo
- `MonthlyCharges` có **phân phối hai đỉnh (bimodal)** — một đỉnh ở ~20$ (gói cơ bản) và một đỉnh ở ~80$ (gói cao cấp). Đây là thách thức cho GAN/VAE vì chúng có xu hướng làm mượt (smooth) phân phối, làm mất một đỉnh

In [3]:
print("="*70)
print("1A. TỔNG QUAN CẤU TRÚC")
print("="*70)
print("Số dòng:", f"{df.shape[0]:,}", "| Số cột:", df.shape[1])
miss = missing_analysis(df)
for col, info in miss['per_column'].items():
    if info['missing_count']>0:
        print(f"  {col:25s}: {info['missing_count']:>5} ({info['missing_pct']:.2f}%)")
if miss['total_missing']==0:
    print("  (Không có missing values)")
dup = duplicate_analysis(df)
print(f"\n  Trùng lặp: {dup['n_exact_duplicates']}")
bal = class_balance(df[target])
print(f"\n--- TARGET ---")
for cls, pct in bal['class_pcts'].items():
    print(f"  {cls:10s}: {bal['class_counts'][cls]:>6} ({pct*100:.1f}%)")
print(f"  Tỷ lệ mất cân bằng: {bal['imbalance_ratio']:.2f}:1")
print(f"  Entropy: {bal['entropy']:.3f} bits (max cho binary = 1 bit)")

1A. TỔNG QUAN CẤU TRÚC
Số dòng: 7,043 | Số cột: 21
  TotalCharges             :    11 (0.16%)

  Trùng lặp: 0

--- TARGET ---
  No        :   5174 (73.5%)
  Yes       :   1869 (26.5%)
  Tỷ lệ mất cân bằng: 2.77:1
  Entropy: 0.835 bits (max cho binary = 1 bit)


In [4]:
print("="*70)
print("1B. THỐNG KÊ MÔ TẢ — BIẾN LIÊN TỤC")
print("="*70)
cont_stats = []
for col in c_cont:
    s=describe_continuous(df[col]); s['name']=col; cont_stats.append(s)
cont_df=pd.DataFrame(cont_stats).set_index('name')
display(cont_df[['count','mean','median','std','min','max','skewness','kurtosis','zero_ratio','iqr']].round(4))

print("\n--- PHÁT HIỆN OUTLIER (3 phương pháp) ---")
rows=[]
for col in c_cont:
    iqr=detect_outliers_iqr(df[col]); zsc=detect_outliers_zscore(df[col]); mad=detect_outliers_mad(df[col])
    rows.append({'col':col,'IQR_n':iqr['n_outliers'],'IQR_pct':f"{iqr['pct_outliers']:.1f}%",
                 'Zscore_n':zsc['n_outliers'],'Zscore_pct':f"{zsc['pct_outliers']:.1f}%",
                 'MAD_n':mad['n_outliers'],'MAD_pct':f"{mad['pct_outliers']:.1f}%"})
display(pd.DataFrame(rows))

print("\n--- ĐA ĐỈNH (MULTIMODALITY) ---")
for col in c_cont:
    m=multimodality_kde_peaks(df[col])
    print(f"  {col:20s}: {m['n_peaks']} đỉnh tại {np.round(m['peak_locations'],2)}")
print("\n--- KIỂM ĐỊNH NORMALITY ---")
for col in c_cont:
    n=normality_test(df[col]); s='NORMAL (p>=0.05)' if n['is_normal'] else 'KHÔNG normal (p<0.05)'
    print(f"  {col:20s}: {n['test_name']:25s} p={n['p_value']:.6f} -> {s}")

1B. THỐNG KÊ MÔ TẢ — BIẾN LIÊN TỤC


,count,mean,median,std,min,max,skewness,kurtosis,zero_ratio,iqr
name,,,,,,,,,,
tenure,7043,32.3711,29.000,24.5595,0.00,72.00,0.2395,-1.3874,0.0016,46.0000
MonthlyCharges,7043,64.7617,70.350,30.0900,18.25,118.75,-0.2205,-1.2573,0.0000,54.3500
TotalCharges,7032,2283.3004,1397.475,2266.7714,18.80,8684.80,0.9616,-0.2318,0.0000,3393.2875



--- PHÁT HIỆN OUTLIER (3 phương pháp) ---


,col,IQR_n,IQR_pct,Zscore_n,Zscore_pct,MAD_n,MAD_pct
0,tenure,0,0.0%,0,0.0%,0,0.0%
1,MonthlyCharges,0,0.0%,0,0.0%,0,0.0%
2,TotalCharges,0,0.0%,0,0.0%,388,5.5%



--- ĐA ĐỈNH (MULTIMODALITY) ---
  tenure              : 2 đỉnh tại [ 4.18 68.39]
  MonthlyCharges      : 3 đỉnh tại [21.27 53.7  81.09]
  TotalCharges        : 1 đỉnh tại [366.13]

--- KIỂM ĐỊNH NORMALITY ---
  tenure              : D'Agostino-Pearson        p=0.000000 -> KHÔNG normal (p<0.05)
  MonthlyCharges      : D'Agostino-Pearson        p=0.000000 -> KHÔNG normal (p<0.05)
  TotalCharges        : D'Agostino-Pearson        p=0.000000 -> KHÔNG normal (p<0.05)


In [5]:

print("="*70)
print(">>> QUYẾT ĐỊNH: SCALING (minmax, standard hay log1p?)")
print("="*70)
for col in c_cont:
    s = describe_continuous(df[col])
    skew = s['skewness'] if s['skewness'] is not None else 0
    has_neg = s['min'] is not None and s['min'] < 0
    if abs(skew) > 1.5 and not has_neg:
        decision = 'log1p'
    elif has_neg:
        decision = 'standard'
    elif abs(skew) > 1.5 and has_neg:
        decision = 'standard'
    else:
        decision = 'minmax'
    if s['count'] == 0: continue
    print(f"  {col:20s}: skew={skew:.2f}, has_neg={has_neg} → {decision}")


>>> QUYẾT ĐỊNH: SCALING (minmax, standard hay log1p?)
  tenure              : skew=0.24, has_neg=False → minmax
  MonthlyCharges      : skew=-0.22, has_neg=False → minmax
  TotalCharges        : skew=0.96, has_neg=False → minmax



**Giải thích quyết định scaling:**
- `tenure` (skew≈0.25, không âm) → minmax
- `MonthlyCharges` (skew≈-0.2) → minmax
- `TotalCharges` (skew=0.96) → minmax
- **Kết luận:** Tất cả minmax. (Config đã cập nhật.)


In [6]:

print("="*70)
print(">>> QUYẾT ĐỊNH: IMPUTATION (mean hay median?)")
print("="*70)
for col in c_cont:
    s = describe_continuous(df[col])
    if s['skewness'] is None: continue
    decision = 'median' if abs(s['skewness']) > 1.0 else 'mean'
    print(f"  {col:20s}: skew={s['skewness']:.2f} → {decision}")


>>> QUYẾT ĐỊNH: IMPUTATION (mean hay median?)
  tenure              : skew=0.24 → mean
  MonthlyCharges      : skew=-0.22 → mean
  TotalCharges        : skew=0.96 → mean



**Giải thích quyết định imputation:**
- `tenure` (skew≈0.25) → mean
- `MonthlyCharges` (skew≈-0.2) → mean
- `TotalCharges` (skew=0.96, gần 1.0) → **median**. *Skew gần 1.0 + có outlier, median an toàn hơn. Đã sửa config từ mean→median.*


In [7]:
print("="*70)
print("1C. THỐNG KÊ MÔ TẢ — BIẾN PHÂN LOẠI")
print("="*70)
cat_rows=[]
for col in c_cat:
    s=describe_categorical(df[col]); ca=cardinality_analysis(df[col])
    cat_rows.append({'col':col,'cardinality':s['cardinality'],'mode':s['mode'],
                     'mode_pct':f"{s['mode_pct']*100:.1f}%",
                     'entropy':round(s['entropy'],3),'n_rare':ca['n_rare']})
display(pd.DataFrame(cat_rows))

1C. THỐNG KÊ MÔ TẢ — BIẾN PHÂN LOẠI


,col,cardinality,mode,mode_pct,entropy,n_rare
0,gender,2,Male,50.5%,1.000,0
1,SeniorCitizen,2,0,83.8%,0.639,0
2,Partner,2,No,51.7%,0.999,0
3,Dependents,2,No,70.0%,0.881,0
4,PhoneService,2,Yes,90.3%,0.459,0
5,MultipleLines,3,No,48.1%,1.359,0
6,InternetService,3,Fiber optic,44.0%,1.529,0
7,OnlineSecurity,3,No,49.7%,1.496,0
8,OnlineBackup,3,No,43.8%,1.529,0
9,DeviceProtection,3,No,43.9%,1.529,0


In [8]:

print("="*70)
print(">>> QUYẾT ĐỊNH: ENCODING (onehot hay label?)")
print("="*70)
for col in c_cat:
    s = describe_categorical(df[col])
    decision = 'onehot' if s['cardinality'] <= 10 else 'label'
    print(f"  {col:20s}: cardinality={s['cardinality']:2d} → {decision}")


>>> QUYẾT ĐỊNH: ENCODING (onehot hay label?)
  gender              : cardinality= 2 → onehot
  SeniorCitizen       : cardinality= 2 → onehot
  Partner             : cardinality= 2 → onehot
  Dependents          : cardinality= 2 → onehot
  PhoneService        : cardinality= 2 → onehot
  MultipleLines       : cardinality= 3 → onehot
  InternetService     : cardinality= 3 → onehot
  OnlineSecurity      : cardinality= 3 → onehot
  OnlineBackup        : cardinality= 3 → onehot
  DeviceProtection    : cardinality= 3 → onehot
  TechSupport         : cardinality= 3 → onehot
  StreamingTV         : cardinality= 3 → onehot
  StreamingMovies     : cardinality= 3 → onehot
  Contract            : cardinality= 3 → onehot
  PaperlessBilling    : cardinality= 2 → onehot
  PaymentMethod       : cardinality= 4 → onehot



**Giải thích quyết định encoding:**
- Tất cả categorical columns trong Telco đều có cardinality ≤ 4 (SeniorCitizen=2, gender=2, Contract=3...)
- → **onehot cho tất cả**. (Config đã cập nhật.)


---
### Dự đoán độ khó cho mô hình sinh — dựa trên thống kê từ Bước 1

| Cột | Chỉ số thống kê | Vấn đề dự đoán cho mô hình sinh | Metric bị ảnh hưởng |
|-----|----------------|--------------------------------|---------------------|
| `TotalCharges` | zero_ratio=11/7043=**0.16%** + skew dương + **functional dependency: tenure × MonthlyCharges** | **GAN/VAE KHÔNG giữ được functional dependency dạng phép nhân.** TotalCharges = tenure × MonthlyCharges. GAN/VAE học phân phối chứ không học phép nhân — có thể sinh ra vô lý (tenure=2 tháng, MonthlyCharges=20$, TotalCharges=5000$). TabDDPM (diffusion) có thể tốt hơn nhờ quá trình denoise có kiểm soát. | **Correlation Diff CAO** — đây là metric phân biệt rõ nhất giữa 3 models cho dataset Telco |
| `MonthlyCharges` | 2 đỉnh (~20$ và ~80$), **bimodal** | **Phân phối hai đỉnh:** GAN/VAE làm mượt (smooth) phân phối → mất một đỉnh. Diffusion giữ được cấu trúc bimodal tốt hơn. | **Wasserstein** |
| `tenure` | skew=**0.25** (gần đối xứng), nhiều đỉnh nhỏ | Tương đối dễ — phân phối gần uniform, không zero-inflation. | **Wasserstein thấp** |
| `SeniorCitizen` | cardinality=2 (0=5901=**83.8%**, 1=1142=**16.2%**) | **Mất cân bằng nghiêm trọng (5:1).** Nếu không explicit khai báo categorical, model sinh ra giá trị 0.3 vô nghĩa. **Đã khai báo trong data_schema.yaml.** | **JSD** |
| `Contract` | 3 categories: Month-to-month=**55%**, One year=**24%**, Two year=**21%** | Tương đối cân bằng, dễ tái tạo. | **JSD thấp** |
| `InternetService` | 3 categories: Fiber optic=**44.2%**, DSL=**34.4%**, No=**21.4%** | Cân bằng, dễ. | **JSD thấp** |
| `PaymentMethod` | 4 categories: Electronic check=**33.6%**, Mailed check=**22.7%**... | Cân bằng, dễ. | **JSD thấp** |
| `MultipleLines` | 3 categories: No=**47.7%**, Yes=**37.5%**, No phone service=**14.8%** | Cân bằng, dễ. | **JSD thấp** |
| Các cột dịch vụ (OnlineSecurity, TechSupport...) | 3 categories: No, Yes, No internet service | "No internet service" là category đặc biệt — model sinh có thể không học được logic "nếu InternetService=No thì các dịch vụ này cũng = No internet service". | **JSD + Correlation Diff** |

---
## Bước 2: Mối quan hệ giữa các biến (Multivariate)

**Mục đích:** Đo lường mức độ phụ thuộc giữa các cặp biến — baseline cho **correlation difference** (metric fidelity). Mối quan hệ càng mạnh thì mô hình sinh càng khó tái tạo chính xác.

- `Contract` và `Churn` có Cramér's V đáng kể — khách hàng Month-to-month có tỷ lệ churn cao hơn hẳn
- `InternetService` và `Churn` — Fiber optic có churn cao hơn DSL
- **TotalCharges = tenure × MonthlyCharges** — đây là functional dependency quan trọng nhất. VIF của TotalCharges sẽ rất cao vì nó là tổ hợp tuyến tính của 2 biến kia
- Các cột dịch vụ (OnlineSecurity, TechSupport...) có quan hệ logic với InternetService — nếu InternetService=No thì tất cả đều = No internet service. Đây là **logic có điều kiện** mà GAN/VAE không học được

In [9]:
print("="*70)
print("2A. CRAMER'S V — Biến phân loại <-> Biến phân loại")
print("="*70)
print("Ý nghĩa: 0=độc lập, 1=phụ thuộc hoàn toàn. >0.3 là đáng kể.")
pairs = [('gender','Churn'),('SeniorCitizen','Churn'),('Partner','Churn'),
         ('Dependents','Churn'),('Contract','Churn'),('InternetService','Churn'),
         ('PaymentMethod','Churn'),('PaperlessBilling','Churn'),
         ('Contract','InternetService'),('PaperlessBilling','PaymentMethod'),
         ('OnlineSecurity','InternetService'),('TechSupport','InternetService')]
for c1,c2 in pairs:
    v=cramers_v(df[c1],df[c2])
    lvl="MẠNH 🟠" if v>0.3 else ("TB 🟡" if v>0.15 else "YẾU 🟢")
    print(f"  V({c1:25s}, {c2:25s}) = {v:.4f}  [{lvl}]")

print("\n--- CORRELATION RATIO η (Phân loại -> Liên tục) ---")
for cat in ['gender','SeniorCitizen','Contract','InternetService','PaymentMethod']:
    for cont in c_cont:
        eta=correlation_ratio(df[cat],df[cont])
        if eta>0.1:
            lvl="MẠNH" if eta>0.25 else "TB"
            print(f"  η({cat:25s} -> {cont:20s}) = {eta:.4f}  [{lvl}]")

print("\n--- THEIL'S U (Uncertainty Coefficient) ---")
for c1,c2 in pairs[:8]:
    u=theils_u(df[c1],df[c2])
    print(f"  U({c1:25s}, {c2:25s}) = {u:.4f}")

2A. CRAMER'S V — Biến phân loại <-> Biến phân loại
Ý nghĩa: 0=độc lập, 1=phụ thuộc hoàn toàn. >0.3 là đáng kể.
  V(gender                   , Churn                    ) = 0.0000  [YẾU 🟢]
  V(SeniorCitizen            , Churn                    ) = 0.1500  [YẾU 🟢]
  V(Partner                  , Churn                    ) = 0.1497  [YẾU 🟢]
  V(Dependents               , Churn                    ) = 0.1634  [TB 🟡]
  V(Contract                 , Churn                    ) = 0.4098  [MẠNH 🟠]
  V(InternetService          , Churn                    ) = 0.3220  [MẠNH 🟠]
  V(PaymentMethod            , Churn                    ) = 0.3027  [MẠNH 🟠]
  V(PaperlessBilling         , Churn                    ) = 0.1911  [TB 🟡]
  V(Contract                 , InternetService          ) = 0.2063  [TB 🟡]
  V(PaperlessBilling         , PaymentMethod            ) = 0.2479  [TB 🟡]
  V(OnlineSecurity           , InternetService          ) = 0.7244  [MẠNH 🟠]
  V(TechSupport              , InternetService       

In [10]:
print("="*70)
print("2B. TƯƠNG QUAN RIÊNG PHẦN + ĐA CỘNG TUYẾN")
print("="*70)
print("--- Tương quan giữa tenure và MonthlyCharges ---")
r_full = df['tenure'].corr(df['MonthlyCharges'], method='spearman')
print(f"  Spearman r(tenure, MonthlyCharges) = {r_full:.4f}")
print("  => tenure và MonthlyCharges có tương quan dương nhẹ (khách ở lâu hơn thường dùng gói đắt hơn)")

print("\n--- VIF (Variance Inflation Factor) ---")
print("VIF > 5: đa cộng tuyến đáng kể. VIF > 10: nghiêm trọng.")
vifs = vif_analysis(df, c_cont)
for col,vif in vifs.items():
    lvl="NGHIÊM TRỌNG 🔴" if vif>10 else ("ĐÁNG KỂ 🟠" if vif>5 else "BÌNH THƯỜNG 🟢")
    print(f"  VIF({col:20s}) = {vif:.2f}  [{lvl}]")
print()
print("=> VIF(TotalCharges) rất cao vì TotalCharges = tenure × MonthlyCharges.")
print("=> Đây là multicollinearity tự nhiên, không phải lỗi dữ liệu.")
print("=> Nhưng Correlation Diff sẽ bị ảnh hưởng nếu model không giữ được functional dependency này.")

2B. TƯƠNG QUAN RIÊNG PHẦN + ĐA CỘNG TUYẾN
--- Tương quan giữa tenure và MonthlyCharges ---
  Spearman r(tenure, MonthlyCharges) = 0.2764
  => tenure và MonthlyCharges có tương quan dương nhẹ (khách ở lâu hơn thường dùng gói đắt hơn)

--- VIF (Variance Inflation Factor) ---
VIF > 5: đa cộng tuyến đáng kể. VIF > 10: nghiêm trọng.
  VIF(tenure              ) = 5.84  [ĐÁNG KỂ 🟠]
  VIF(MonthlyCharges      ) = 3.23  [BÌNH THƯỜNG 🟢]
  VIF(TotalCharges        ) = 9.53  [ĐÁNG KỂ 🟠]

=> VIF(TotalCharges) rất cao vì TotalCharges = tenure × MonthlyCharges.
=> Đây là multicollinearity tự nhiên, không phải lỗi dữ liệu.
=> Nhưng Correlation Diff sẽ bị ảnh hưởng nếu model không giữ được functional dependency này.


In [11]:
print("="*70)
print("2C. MUTUAL INFORMATION VỚI TARGET (bits)")
print("="*70)
mi_vs_target = {}
for col in c_cont + c_cat:
    is_x_num = col in c_cont
    mi = mutual_information(df[col], df[target], discrete_x=not is_x_num, discrete_y=True)
    mi_vs_target[col] = mi
for col,mi in sorted(mi_vs_target.items(), key=lambda x:-x[1]):
    lvl = "RẤT CAO 🔴" if mi>0.3 else ("CAO 🟠" if mi>0.1 else ("TB 🟡" if mi>0.05 else "THẤP 🟢"))
    print(f"  MI({col:25s}, Churn) = {mi:.4f} bits  [{lvl}]")

2C. MUTUAL INFORMATION VỚI TARGET (bits)
  MI(Contract                 , Churn) = 0.1420 bits  [CAO 🟠]
  MI(tenure                   , Churn) = 0.1127 bits  [CAO 🟠]
  MI(OnlineSecurity           , Churn) = 0.0933 bits  [TB 🟡]
  MI(TechSupport              , Churn) = 0.0909 bits  [TB 🟡]
  MI(InternetService          , Churn) = 0.0802 bits  [TB 🟡]
  MI(OnlineBackup             , Churn) = 0.0675 bits  [TB 🟡]
  MI(TotalCharges             , Churn) = 0.0643 bits  [TB 🟡]
  MI(PaymentMethod            , Churn) = 0.0642 bits  [TB 🟡]
  MI(DeviceProtection         , Churn) = 0.0634 bits  [TB 🟡]
  MI(MonthlyCharges           , Churn) = 0.0591 bits  [TB 🟡]
  MI(StreamingMovies          , Churn) = 0.0462 bits  [THẤP 🟢]
  MI(StreamingTV              , Churn) = 0.0460 bits  [THẤP 🟢]
  MI(PaperlessBilling         , Churn) = 0.0277 bits  [THẤP 🟢]
  MI(Dependents               , Churn) = 0.0209 bits  [THẤP 🟢]
  MI(Partner                  , Churn) = 0.0165 bits  [THẤP 🟢]
  MI(SeniorCitizen            , 

---
### Dự đoán độ khó — dựa trên Multivariate Statistics

| Mối quan hệ | Chỉ số | Mức khó | Giải thích |
|------------|--------|---------|------------|
| V(Contract, Churn) | Cramér's V | 🟠 **Trung bình** | Khách hàng Month-to-month có churn cao hơn hẳn Two year. CTGAN có xu hướng làm yếu association này. |
| V(InternetService, Churn) | Cramér's V | 🟠 **Trung bình** | Fiber optic có churn cao hơn DSL. |
| V(OnlineSecurity, InternetService) | Cramér's V | 🟠 **Trung bình-Cao** | **Logic có điều kiện:** nếu InternetService=No thì OnlineSecurity=No internet service. GAN/VAE không học được logic này. |
| **TotalCharges = tenure × MonthlyCharges** | Functional dependency | 🔴 **CỰC KỲ KHÓ** | GAN/VAE học phân phối chứ không học phép nhân. TabDDPM có thể tốt hơn. **Đây là phép thử phân biệt chính cho Telco.** |
| VIF(TotalCharges) **cao** | Multicollinearity | 🔴 **Nghiêm trọng** | Vì TotalCharges là tổ hợp của tenure và MonthlyCharges. Correlation Diff sẽ cao. |
| MI(Tenure, Churn) | Mutual Information | 🟠 **Cao** | tenure là feature quan trọng nhất để predict Churn. Nếu synthetic không tái tạo đúng tenure → Utility giảm. |

---
## Bước 3: Cảnh báo đặc biệt cho Evaluation

**Góc nhìn khoa học dữ liệu:**
- **Telco KHÔNG có ceiling effect mạnh như Adult** — không có feature nào definitional overlap với Churn
- **Nhưng có functional dependency (TotalCharges)** — đây là thách thức lớn nhất cho Fidelity
- **Logic có điều kiện giữa InternetService và các dịch vụ con** — GAN/VAE không học được
- **SeniorCitizen mất cân bằng 5:1** — model sinh thiếu lớp 1

In [12]:
print("="*70)
print("CEILING EFFECT? — Telco không có definitional overlap")
print("="*70)
print("Kiểm tra: TotalCharges có 'biết trước' Churn không?")
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
X = df[['tenure','MonthlyCharges','TotalCharges']].fillna(0)
y = (df['Churn']=='Yes').astype(int)
scores = cross_val_score(RandomForestClassifier(n_estimators=50, random_state=42), X, y, cv=3)
print(f"  Cross-val accuracy (chỉ 3 continuous features): {scores.mean():.3f} (±{scores.std():.3f})")
print("  => TotalCharges có chứa thông tin về Churn, nhưng không definitional overlap.")
print("  => Không cần ablation study cho Telco.")

print("\n--- MODE COLLAPSE RISK ---")
for col in c_cat:
    ca=cardinality_analysis(df[col])
    if ca['n_rare']>0:
        print(f"  {col}: {ca['n_rare']} categories hiếm (VD: {', '.join(ca['rare_categories'][:3])})")

print("\n--- MEMORIZATION RISK (outlier cực đoan) ---")
print(f"  MonthlyCharges >= 115: {(df['MonthlyCharges']>=115).sum()} samples ({(df['MonthlyCharges']>=115).mean()*100:.2f}%)")
print(f"  TotalCharges >= 8000: {(df['TotalCharges']>=8000).sum()} samples ({(df['TotalCharges']>=8000).mean()*100:.2f}%)")
print("  => Tỷ lệ rất thấp, nguy cơ memorization thấp.")

print("\n--- COVARIATE SHIFT (Train vs Test) ---")
from sklearn.model_selection import train_test_split
df_tr,df_te = train_test_split(df, test_size=0.2, random_state=42)
drift = covariate_shift_detection(df_tr, df_te, c_cont, c_cat)
n_drift = sum(1 for d in drift if d['p_value']<0.05)
print(f"  Features có drift (p<0.05): {n_drift}/{len(drift)}")
for d in drift:
    if d['p_value']<0.05:
        print(f"  ** {d['feature']:25s}: {d['method']:25s} p={d['p_value']:.4f}")
print("  => stratified split đã dùng → giảm drift. Kiểm tra lại sau pipeline.")

CEILING EFFECT? — Telco không có definitional overlap
Kiểm tra: TotalCharges có 'biết trước' Churn không?
  Cross-val accuracy (chỉ 3 continuous features): 0.761 (±0.003)
  => TotalCharges có chứa thông tin về Churn, nhưng không definitional overlap.
  => Không cần ablation study cho Telco.

--- MODE COLLAPSE RISK ---

--- MEMORIZATION RISK (outlier cực đoan) ---
  MonthlyCharges >= 115: 65 samples (0.92%)
  TotalCharges >= 8000: 78 samples (1.11%)
  => Tỷ lệ rất thấp, nguy cơ memorization thấp.

--- COVARIATE SHIFT (Train vs Test) ---
  Features có drift (p<0.05): 2/19
  ** gender                   : Chi-square homogeneity    p=0.0110
  ** Contract                 : Chi-square homogeneity    p=0.0424
  => stratified split đã dùng → giảm drift. Kiểm tra lại sau pipeline.


---
## Bước 4: Baseline Association Matrix

**Đây là "ground truth" — correlation difference trong evaluation sẽ so sánh synthetic với các giá trị này.**

In [13]:
print("=== BASELINE CRAMER'S V (Real Data) ===")
print("(synthetic data cần đạt càng gần các giá trị này càng tốt)")
cat_cols = ['gender','SeniorCitizen','Contract','InternetService','PaymentMethod','Churn']
for c1 in cat_cols:
    for c2 in cat_cols:
        if c1<c2:
            v=cramers_v(df[c1],df[c2])
            print(f"  V({c1:25s}, {c2:25s}) = {v:.4f}")
print()
print("=> Correlation difference = mean absolute diff giữa baseline và synthetic.")
print("   Nếu overall diff < 0.05: fidelity rất tốt.")
print("   Nếu diff > 0.10: model sinh không giữ được cấu trúc tương quan.")

=== BASELINE CRAMER'S V (Real Data) ===
(synthetic data cần đạt càng gần các giá trị này càng tốt)
  V(SeniorCitizen            , gender                   ) = 0.0000
  V(Contract                 , gender                   ) = 0.0000
  V(Contract                 , SeniorCitizen            ) = 0.1429
  V(Contract                 , InternetService          ) = 0.2063
  V(Contract                 , PaymentMethod            ) = 0.2659
  V(InternetService          , gender                   ) = 0.0000
  V(InternetService          , SeniorCitizen            ) = 0.2648
  V(InternetService          , PaymentMethod            ) = 0.3125
  V(PaymentMethod            , gender                   ) = 0.0000
  V(PaymentMethod            , SeniorCitizen            ) = 0.1949
  V(Churn                    , gender                   ) = 0.0000
  V(Churn                    , SeniorCitizen            ) = 0.1500
  V(Churn                    , Contract                 ) = 0.4098
  V(Churn                    ,

---
## Bước 5: Dự đoán kết quả và Hạn chế

### Dự đoán chi tiết (dựa trên số liệu thống kê cụ thể từ B1-B4)

#### Fidelity
| Metric | CTGAN | TVAE | TabDDPM | Cơ sở thống kê |
|--------|-------|------|---------|---------------|
| **Correlation Diff** | **Cao nhất** | Trung bình | **Thấp nhất** | TotalCharges = tenure × MonthlyCharges. GAN/VAE không giữ được functional dependency. **Đây là metric phân biệt rõ nhất cho Telco.** |
| Wasserstein (MonthlyCharges) | TB | TB | Thấp | Bimodal distribution (2 đỉnh ở 20$ và 80$). Diffusion giữ bimodal tốt hơn. |
| Wasserstein (tenure) | Thấp | Thấp | Thấp | Gần uniform, nhiều đỉnh nhỏ — model nào cũng tái tạo tốt. |
| JSD (SeniorCitizen) | **Cao** | TB | **Thấp** | Mất cân bằng 5:1 (83.8% là 0). CTGAN smooth → mất tỷ lệ thật. |
| JSD (Contract) | TB | TB | Thấp | 3 categories cân bằng. |
| JSD (InternetService) | TB | TB | Thấp | 3 categories cân bằng. |

#### Privacy
| Metric | CTGAN | TVAE | TabDDPM | Cơ sở thống kê |
|--------|-------|------|---------|---------------|
| DCR Mean | **Cao nhất (tốt nhất)** | TB | Thấp nhất | CTGAN sinh ngẫu nhiên hơn → xa real data hơn. Diffusion memorization → gần hơn. |
| MIA AUC | **Thấp nhất (an toàn)** | TB | Cao nhất | Diffusion memorizes training data → attacker phân biệt member vs non-member dễ hơn. |

#### Utility
| Metric | Ghi chú |
|--------|---------|
| TSTR F1 | **Không có ceiling effect mạnh.** Kết quả Utility phản ánh đúng chất lượng synthetic. |
| Ablation | **Không cần** — không có definitional overlap. |

### ⚠️ Hạn chế cần ghi nhận (Limitations)
1. **Functional dependency (TotalCharges = tenure × MonthlyCharges):** Đây là phép thử phân biệt chính giữa 3 models. Nếu correlation diff thấp cho cả 3 → cả 3 đều giữ được functional dependency (khả năng thấp).
2. **SeniorCitizen imbalance (83.8% là 0):** Model sinh thiếu lớp 1. Cần kiểm tra JSD cho SeniorCitizen.
3. **Logic có điều kiện:** InternetService=No → tất cả dịch vụ con = No internet service. GAN/VAE không học được logic này.
4. **TotalCharges missing (11 rows):** Xử lý bằng median imputation. Ảnh hưởng không đáng kể (0.16%).

In [14]:
print("="*70)
print("TỔNG HỢP — EDA KEY NUMBERS")
print("="*70)
bal=class_balance(df[target])
miss=missing_analysis(df)
dup=duplicate_analysis(df)
vifs=vif_analysis(df,c_cont)
print(f"  Target imbalance ratio:       {bal['imbalance_ratio']:.2f}:1")
print(f"  Missing cells:                {miss['total_missing']:,}/{miss['total_cells']:,} ({miss['overall_missing_pct']:.2f}%)")
print(f"  Duplicate rows:               {dup['n_exact_duplicates']}")
print(f"  Zero-inflation > 50%:         {sum(1 for c in c_cont if describe_continuous(df[c])['zero_ratio']>0.5)}/{len(c_cont)}")
print(f"  Features not normal:          {sum(1 for c in c_cont if not normality_test(df[c])['is_normal'])}/{len(c_cont)}")
print(f"  VIF > 5:                      {sum(1 for v in vifs.values() if v>5)}/{len(c_cont)}")
print(f"  Categorical with rare cats:   {sum(1 for c in c_cat if cardinality_analysis(df[c])['n_rare']>0)}/{len(c_cat)}")
print()
print(">> Các con số này là BASELINE để so sánh với synthetic data.")

TỔNG HỢP — EDA KEY NUMBERS
  Target imbalance ratio:       2.77:1
  Missing cells:                11/147,903 (0.01%)
  Duplicate rows:               0
  Zero-inflation > 50%:         0/3
  Features not normal:          3/3
  VIF > 5:                      2/3
  Categorical with rare cats:   0/16

>> Các con số này là BASELINE để so sánh với synthetic data.


---
## Tài liệu tham khảo
1. **Kaggle Telco Customer Churn:** https://www.kaggle.com/datasets/blastchar/telco-customer-churn
2. **Wasserstein-1:** đo khác biệt 2 phân phối liên tục, normalized bằng MinMax của real data range
3. **JSD (base=2):** Jensen-Shannon Divergence, đo khác biệt 2 phân phối categorical, return ∈ [0,1]
4. **Cramér's V (bias-corrected):** Bergsma & Wicher (2013) — hiệu chỉnh thiên vị cho mẫu nhỏ
5. **Theil's U:** uncertainty coefficient dựa trên mutual information
6. **Functional dependency trong synthetic data:** TotalCharges = tenure × MonthlyCharges — phép thử cho correlation diff